In [0]:
-- ============================================================================
-- MegaMart Databricks Capstone
-- Stage: 04 - Sales Analysis
--
-- Purpose:
--   Analyze cleaned Silver sales data and create dashboard-ready Gold
--   datasets for store and product sales performance.
--
-- Source:
--   megamart_dev.silver.*
--
-- Target:
--   megamart_dev.gold.*
--
-- Important:
--   The data-generation script is the source of truth for the actual
--   transaction schema. Sales transactions contain:
--       transaction_id
--       store_id
--       product_id
--       quantity_sold
--       unit_price
--       total_amount
--       sale_date
--
--   True profitability is NOT calculated because the generated source
--   contains selling price but no product unit cost.
-- ============================================================================


-- ============================================================================
-- 1. Select Project Catalog
-- ============================================================================

USE CATALOG megamart_dev;


-- ============================================================================
-- 2. Verify Silver Source Counts
-- ============================================================================

SELECT
    'sales_transactions' AS dataset,
    COUNT(*) AS record_count
FROM silver.sales_transactions

UNION ALL

SELECT
    'stores' AS dataset,
    COUNT(*) AS record_count
FROM silver.stores

UNION ALL

SELECT
    'products' AS dataset,
    COUNT(*) AS record_count
FROM silver.products;

-- ============================================================================
-- 9. Sales Analysis Verification
-- ============================================================================

SELECT
    'store_performance' AS view_name,
    COUNT(*) AS record_count
FROM gold.store_performance

UNION ALL

SELECT
    'top_products' AS view_name,
    COUNT(*) AS record_count
FROM gold.top_products

UNION ALL

SELECT
    'monthly_store_trends' AS view_name,
    COUNT(*) AS record_count
FROM gold.monthly_store_trends

UNION ALL

SELECT
    'product_performance' AS view_name,
    COUNT(*) AS record_count
FROM gold.product_performance;


-- ============================================================================
-- 10. Store Revenue Results
-- ============================================================================

SELECT
    store_id,
    store_name,
    location,
    total_revenue,
    transaction_count,
    average_transaction_value,
    unique_products_sold

FROM gold.store_performance

ORDER BY total_revenue DESC;


-- ============================================================================
-- 11. Top 5 Products
-- ============================================================================

SELECT
    quantity_rank,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    total_revenue,
    transaction_count

FROM gold.top_products

ORDER BY quantity_rank;


-- ============================================================================
-- 12. Store Rankings
-- ============================================================================

SELECT
    store_id,
    store_name,
    total_revenue,
    transaction_count,
    revenue_rank,
    transaction_count_rank

FROM gold.store_performance

ORDER BY
    revenue_rank,
    store_id;


-- ============================================================================
-- 13. Monthly Store Trend
-- ============================================================================

SELECT
    store_id,
    store_name,
    year_month,
    monthly_revenue,
    previous_month_revenue,
    mom_revenue_change,
    mom_revenue_growth_pct

FROM gold.monthly_store_trends

ORDER BY
    store_id,
    month_start;


-- ============================================================================
-- 14. Product Diversity
-- ============================================================================

SELECT
    store_id,
    store_name,
    location,
    unique_products_sold,
    unique_categories_sold

FROM gold.store_performance

ORDER BY unique_products_sold DESC;


-- ============================================================================
-- 15. Record Sales Analysis Stage in Audit Table
-- ============================================================================

INSERT INTO audit.pipeline_runs
SELECT
    CONCAT(
        '04_sales_',
        DATE_FORMAT(
            CURRENT_TIMESTAMP(),
            'yyyyMMddHHmmss'
        )
    ) AS run_id,

    'megamart_pipeline' AS pipeline_name,

    '04_sales' AS stage_name,

    'SUCCESS' AS status,

    CURRENT_TIMESTAMP() AS start_time,

    CURRENT_TIMESTAMP() AS end_time,

    (
        SELECT COUNT(*)
        FROM silver.sales_transactions
    ) AS records_processed,

    0 AS records_failed,

    NULL AS error_message;


-- ============================================================================
-- End of Stage 04
-- ============================================================================